In [1]:
import os 

In [2]:
%pwd

'd:\\projects\\Text Summarizer\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\projects\\Text Summarizer'

In [5]:
# model_trainer.py
from dataclasses import dataclass
from pathlib import Path
import time
from tqdm import tqdm
import psutil
import torch
import os
import gc

from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq,
    TrainerCallback
)
from datasets import load_from_disk


@dataclass
class ModelTrainerConfig:
    root_dir: str
    data_path: str
    model_ckpt: str
    num_train_epochs: int = 1
    warmup_steps: int = 20
    per_device_train_batch_size: int = 1
    weight_decay: float = 0.01
    logging_steps: int = 5
    eval_steps: int = 20
    save_steps: int = 40
    gradient_accumulation_steps: int = 8
    max_input_length: int = 128
    max_target_length: int = 32
    subset_percentage: int = 5


class ModelTrainer:
    def __init__(self, config):
        self.config = config
    
    def find_optimal_batch_size(self, model, tokenizer, dataset_sample):
        """Dynamically find the largest batch size that fits in GPU memory"""
        print("\n🔍 Finding optimal batch size for your GPU...")
        
        if not torch.cuda.is_available():
            print("  No GPU found, using batch size 1")
            return 1
        
        test_batch_sizes = [1, 2, 4]
        optimal_batch_size = 1
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        print(f"  GPU Memory: {total_memory:.2f} GB")
        
        for batch_size in test_batch_sizes:
            try:
                gc.collect()
                torch.cuda.empty_cache()
                
                sample = dataset_sample[:batch_size]
                
                inputs = tokenizer(
                    sample['dialogue'],
                    max_length=self.config.max_input_length,
                    truncation=True,
                    padding="max_length",
                    return_tensors="pt"
                ).to('cuda')
                
                labels = tokenizer(
                    sample['summary'],
                    max_length=self.config.max_target_length,
                    truncation=True,
                    padding="max_length",
                    return_tensors="pt"
                ).to('cuda')
                
                inputs['labels'] = labels['input_ids']
                
                with torch.cuda.amp.autocast():
                    outputs = model(**inputs)
                    loss = outputs.loss
                
                loss.backward()
                
                allocated = torch.cuda.memory_allocated() / 1e9
                print(f"  ✓ Batch size {batch_size}: {allocated:.2f}GB used")
                
                if allocated > total_memory * 0.8:
                    print(f"  ⚠ Memory usage too high (>80%), stopping")
                    optimal_batch_size = batch_size - 1 if batch_size > 1 else 1
                    break
                
                optimal_batch_size = batch_size
                
                del inputs, labels, outputs, loss
                gc.collect()
                torch.cuda.empty_cache()
                
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  ✗ Batch size {batch_size} failed: OOM")
                    optimal_batch_size = batch_size - 1 if batch_size > 1 else 1
                    break
                else:
                    print(f"  ✗ Batch size {batch_size} failed: {e}")
                    break
        
        print(f"\n✅ Optimal batch size: {optimal_batch_size}")
        return optimal_batch_size
    
    class ProgressCallback(TrainerCallback):
        def __init__(self, total_steps):
            self.start_time = time.time()
            self.total_steps = total_steps
            self.progress_bar = tqdm(total=total_steps, desc="Training", position=0, mininterval=1.0)
        
        def on_step_end(self, args, state, control, **kwargs):
            self.progress_bar.update(1)
            if state.global_step % 10 == 0:
                elapsed = time.time() - self.start_time
                steps_per_sec = state.global_step / elapsed if elapsed > 0 else 0
                self.progress_bar.set_postfix({
                    'Step': f"{state.global_step}/{self.total_steps}",
                    'Speed': f"{steps_per_sec:.2f} steps/s",
                })
        
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and 'loss' in logs:
                self.progress_bar.set_postfix({'Loss': f"{logs['loss']:.4f}"})
        
        def on_train_end(self, args, state, control, **kwargs):
            self.progress_bar.close()
    
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
        if device == "cuda":
            print(f"\n{'='*60}")
            print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
            print(f"   Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
            print(f"{'='*60}")
        
        print(f"\n🔄 Loading model...")
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt,
            low_cpu_mem_usage=True,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        model = model.to(device)
        model.gradient_checkpointing_enable()
        
        print(f"\n🔄 Loading dataset sample...")
        dataset = load_from_disk(self.config.data_path)
        sample_size = min(16, len(dataset['train']))
        dataset_sample = dataset['train'].select(range(sample_size))
        
        per_device_batch_size = self.config.per_device_train_batch_size
        if device == "cuda":
            per_device_batch_size = self.find_optimal_batch_size(model, tokenizer, dataset_sample)
        
        print(f"\n🔄 Loading full dataset...")
        train_size = int(len(dataset['train']) * self.config.subset_percentage / 100)
        dataset['train'] = dataset['train'].select(range(train_size))
        dataset['test'] = dataset['test'].select(range(min(10, len(dataset['test']))))
        print(f"  Training on {len(dataset['train'])} samples")
        
        print("\n🔄 Tokenizing...")
        def preprocess_function(examples):
            model_inputs = tokenizer(
                examples['dialogue'], 
                max_length=self.config.max_input_length,
                truncation=True, 
                padding="max_length"
            )
            labels = tokenizer(
                examples['summary'], 
                max_length=self.config.max_target_length,
                truncation=True, 
                padding="max_length"
            )
            model_inputs["labels"] = [
                [id if id != tokenizer.pad_token_id else -100 for id in label]
                for label in labels["input_ids"]
            ]
            return model_inputs
        
        tokenized_dataset = dataset.map(
            preprocess_function, 
            batched=True,
            batch_size=25,
            remove_columns=dataset['train'].column_names,
            desc="Tokenizing",
        )
        
        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model,
            padding=True,
            label_pad_token_id=-100,
        )
        
        gradient_accumulation = self.config.gradient_accumulation_steps
        total_steps = (len(tokenized_dataset['train']) // per_device_batch_size) * self.config.num_train_epochs
        
        # Build arguments dynamically (removed save_safetensors)
        training_args_kwargs = {
            "output_dir": self.config.root_dir,
            "num_train_epochs": self.config.num_train_epochs,
            "per_device_train_batch_size": per_device_batch_size,
            "per_device_eval_batch_size": 1,
            "warmup_steps": self.config.warmup_steps,
            "weight_decay": self.config.weight_decay,
            "logging_dir": os.path.join(self.config.root_dir, "logs"),
            "logging_steps": self.config.logging_steps,
            "save_strategy": "steps",
            "save_steps": self.config.save_steps,
            "save_total_limit": 1,
            "load_best_model_at_end": False,
            "predict_with_generate": False,
            "generation_max_length": self.config.max_target_length,
            "generation_num_beams": 1,
            "report_to": "none",
            "gradient_accumulation_steps": gradient_accumulation,
            "dataloader_num_workers": 0,
            "optim": "adamw_torch",
            "lr_scheduler_type": "linear",
            "seed": 42,
            "disable_tqdm": False,
            "dataloader_pin_memory": False,
            "max_grad_norm": 1.0,
        }
        
        if device == "cuda":
            training_args_kwargs["fp16"] = True
        
        # Use eval_strategy (detected earlier)
        training_args_kwargs["eval_strategy"] = "steps"
        training_args_kwargs["eval_steps"] = self.config.eval_steps
        
        # Create training arguments
        training_args = Seq2SeqTrainingArguments(**training_args_kwargs)
        
        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            eval_dataset=tokenized_dataset["test"],
            data_collator=data_collator,
        )
        
        trainer.add_callback(self.ProgressCallback(total_steps))
        
        print("\n" + "="*60)
        print("🚀 FINAL TRAINING CONFIGURATION")
        print("="*60)
        print(f"  Batch size: {per_device_batch_size}")
        print(f"  Gradient accumulation: {gradient_accumulation}")
        print(f"  Effective batch size: {per_device_batch_size * gradient_accumulation}")
        print(f"  Training samples: {len(tokenized_dataset['train'])}")
        print(f"  Total steps: {total_steps}")
        print("="*60)
        
        train_start = time.time()
        try:
            trainer.train()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"\n❌ OOM Error during training")
                print(f"   Try reducing batch size to {per_device_batch_size - 1}")
            raise e
        
        train_time = time.time() - train_start
        print(f"\n✓ Training completed in {train_time/60:.2f} minutes!")
        
        print(f"\n💾 Saving model...")
        os.makedirs(self.config.root_dir, exist_ok=True)
        trainer.save_model(self.config.root_dir)
        tokenizer.save_pretrained(self.config.root_dir)
        
        saved_files = os.listdir(self.config.root_dir)
        print(f"\n📁 Files saved:")
        for file in saved_files[:5]:
            file_path = os.path.join(self.config.root_dir, file)
            if os.path.isfile(file_path):
                size = os.path.getsize(file_path) / (1024*1024)
                print(f"  • {file}: {size:.2f} MB")
        
        return trainer


if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 AUTO-OPTIMIZING MODEL TRAINER")
    print("="*60)
    
    class DirectConfig:
        def __init__(self):
            self.root_dir = r"d:/projects/Text Summarizer/notebooks/model_trainer"
            self.data_path = r"d:/projects/Text Summarizer/artifacts/data_transformation/samsum_dataset"
            self.model_ckpt = "google/pegasus-cnn_dailymail"
            self.num_train_epochs = 1
            self.warmup_steps = 20
            self.per_device_train_batch_size = 1
            self.weight_decay = 0.01
            self.logging_steps = 5
            self.eval_steps = 20
            self.save_steps = 40
            self.gradient_accumulation_steps = 8
            self.max_input_length = 128
            self.max_target_length = 32
            self.subset_percentage = 5
    
    config = DirectConfig()
    
    if not os.path.exists(config.data_path):
        print(f"\n❌ Data path not found: {config.data_path}")
        exit(1)
    
    os.makedirs(config.root_dir, exist_ok=True)
    
    trainer = ModelTrainer(config)
    trainer.train()


🚀 AUTO-OPTIMIZING MODEL TRAINER

🔄 Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🔄 Loading dataset sample...

🔄 Loading full dataset...
  Training on 736 samples

🔄 Tokenizing...


Tokenizing:   0%|          | 0/818 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Training:   0%|          | 0/736 [00:00<?, ?it/s]


🚀 FINAL TRAINING CONFIGURATION
  Batch size: 1
  Gradient accumulation: 8
  Effective batch size: 8
  Training samples: 736
  Total steps: 736


: 